In [131]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

project_root = Path.cwd().resolve().parent.parent
season_dir = project_root / 'src' / 'data' / 'raw' / '2025'
processed_dir = project_root / 'src' / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

races = sorted(season_dir.iterdir())
train_races = set(races[:12])
test_races  = set(races[12:])

train_dfs = []
test_dfs  = []

for race_folder in races:
    if not race_folder.is_dir():
        continue
    matches = list(race_folder.glob('*_R_laps.csv'))
    if not matches:
        print(f"Skipped {race_folder.name} — no laps file found")
        continue
    try:
        df_race = pd.read_csv(matches[0])
        df_race['RaceName'] = race_folder.name
        if race_folder in train_races:
            train_dfs.append(df_race)
        else:
            test_dfs.append(df_race)
        print(f"Loaded {race_folder.name} — {len(df_race)} laps")
    except Exception as e:
        print(f"Error loading {race_folder.name}: {e}")

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs,  ignore_index=True)

print(f"\nTrain: {len(df_train)} laps across {len(train_dfs)} races")
print(f"Test:  {len(df_test)} laps across {len(test_dfs)} races")

Loaded Abu_Dhabi_Grand_Prix — 1156 laps
Loaded Australian_Grand_Prix — 927 laps
Loaded Austrian_Grand_Prix — 1126 laps
Loaded Azerbaijan_Grand_Prix — 968 laps
Loaded Bahrain_Grand_Prix — 1128 laps
Loaded Belgian_Grand_Prix — 879 laps
Loaded British_Grand_Prix — 825 laps
Loaded Canadian_Grand_Prix — 1349 laps
Loaded Chinese_Grand_Prix — 1065 laps
Loaded Dutch_Grand_Prix — 1364 laps
Loaded Emilia_Romagna_Grand_Prix — 1207 laps
Loaded Hungarian_Grand_Prix — 1368 laps
Loaded Italian_Grand_Prix — 974 laps
Loaded Japanese_Grand_Prix — 1059 laps
Loaded Las_Vegas_Grand_Prix — 886 laps
Loaded Mexico_City_Grand_Prix — 1263 laps
Loaded Miami_Grand_Prix — 1005 laps
Loaded Monaco_Grand_Prix — 1425 laps
Loaded Qatar_Grand_Prix — 1067 laps
Loaded Saudi_Arabian_Grand_Prix — 898 laps
Loaded Singapore_Grand_Prix — 1229 laps
Loaded Spanish_Grand_Prix — 1203 laps
Loaded São_Paulo_Grand_Prix — 1251 laps
Loaded United_States_Grand_Prix — 1067 laps

Train: 13362 laps across 12 races
Test:  13327 laps across 

In [132]:
def preprocess(df, le_driver=None, le_team=None, le_compound=None, fit=False):

    df = df.copy()

    # Create target
    df['PitLap'] = df['PitInTime'].notna().astype(int)

    # Convert timedelta columns to float seconds
    time_cols = ['LapTime', 'PitOutTime', 'PitInTime',
                 'Sector1Time', 'Sector2Time', 'Sector3Time',
                 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
                 'LapStartTime', 'Time']
    for col in time_cols:
        df[col] = pd.to_timedelta(df[col]).dt.total_seconds()

    # Encode categorical, fit on train, transform on test
    if fit:
        le_driver = LabelEncoder()
        le_team = LabelEncoder()
        le_compound = LabelEncoder()
        df['Driver'] = le_driver.fit_transform(df['Driver'].astype(str))
        df['Team'] = le_team.fit_transform(df['Team'].astype(str))
        df['Compound'] = le_compound.fit_transform(df['Compound'].astype(str))
    else:
        df['Driver'] = le_driver.transform(df['Driver'].astype(str))
        df['Team'] = le_team.transform(df['Team'].astype(str))
        df['Compound'] = le_compound.transform(df['Compound'].astype(str))

    # Boolean columns
    df['FreshTyre'] = df['FreshTyre'].astype(int)
    df['IsPersonalBest'] = df['IsPersonalBest'].fillna(0).astype(int)
    df['IsAccurate'] = df['IsAccurate'].astype(int)

    # Sort by driver and lap to ensure correct order
    df = df.sort_values(['Driver', 'LapNumber'])

    # Speed degradation over last 3 laps, is the car getting slower?
    df['SpeedI1_delta'] = df.groupby('Driver')['SpeedI1'].diff()
    df['SpeedST_delta'] = df.groupby('Driver')['SpeedST'].diff()

    # Laps since last pit, how deep into the stint are we?
    df['LapInStint'] = df.groupby('Driver').cumcount()

    # Rolling average speed, smooths out noise
    df['SpeedI1_roll3'] = df.groupby('Driver')['SpeedI1'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
)

    # Create target
    df['PitLap'] = df['PitInTime'].notna().astype(int)

    # Then drop the leaky columns
    drop_cols = ['LapStartDate', 'TrackStatus', 'Deleted',
             'DeletedReason', 'FastF1Generated', 'DriverNumber',
             'RaceName', 'PitInTime', 'PitOutTime',
             'LapTime', 'Stint',
             'Sector1Time', 'Sector2Time', 'Sector3Time',
             'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'IsPersonalBest', 'IsAccurate', 'LapStartTime', 'SpeedFL', 'Time']

    df.drop(columns=drop_cols, inplace=True)

    # Remove all NaNs
    df.fillna(0, inplace=True)

    # Verify all NaNs have been removed
    assert df.isna().sum().sum() == 0, "NaNs still present"

    return df, le_driver, le_team, le_compound

In [133]:
# Fit encoders on train, apply same to test
df_train, le_driver, le_team, le_compound = preprocess(df_train, fit=True)
df_test,_,_,_= preprocess(df_test,le_driver=le_driver,le_team=le_team,le_compound=le_compound, fit=False)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Train shape: (13362, 15)
Test shape:  (13327, 15)


In [134]:
meta_train = df_train[['Driver', 'LapNumber']].copy()
meta_test = df_test[['Driver', 'LapNumber']].copy()

meta_train.to_csv(processed_dir / 'meta_train.csv', index=False)
meta_test.to_csv(processed_dir  / 'meta_test.csv',  index=False)

# Save driver mapping for readable names later
driver_mapping = {int(code): name for code, name in enumerate(le_driver.classes_)}
with open(processed_dir / 'driver_mapping.json', 'w') as f:
    json.dump(driver_mapping, f)

In [135]:
target = 'PitLap'
feature_cols = [c for c in df_train.columns if c != target]

X_train = df_train[feature_cols].values.astype(np.float32)
y_train = df_train[target].values.astype(np.float32).reshape(-1, 1)

X_test  = df_test[feature_cols].values.astype(np.float32)
y_test  = df_test[target].values.astype(np.float32).reshape(-1, 1)

# Fit scaler on train only, apply to test
scaler_X = StandardScaler()
X_train  = scaler_X.fit_transform(X_train)
X_test   = scaler_X.transform(X_test)

# SMOTE balancing for class balancing
sm = SMOTE(random_state=42)
X_train, y_train_flat = sm.fit_resample(X_train, y_train.ravel())
y_train = y_train_flat.reshape(-1, 1)

print(f"After SMOTE X_train: {X_train.shape}")
print(f"Pit laps in train: {y_train.sum().astype(int)}")
print(f"Non-pit laps in train: {(y_train == 0).sum().astype(int)}")

# Save arrays
np.save(processed_dir / 'X_train.npy', X_train)
np.save(processed_dir / 'X_test.npy',  X_test)
np.save(processed_dir / 'y_train.npy', y_train)
np.save(processed_dir / 'y_test.npy',  y_test)

# Displaying the processed data
print(feature_cols)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}") # Test set has slightly more laps than training due to race splits
print(f"Pit laps in train: {y_train.sum().astype(int)}")
print(f"Pit laps in test:  {y_test.sum().astype(int)}")

After SMOTE X_train: (25756, 14)
Pit laps in train: 12878
Non-pit laps in train: 12878
['Driver', 'LapNumber', 'SpeedI1', 'SpeedI2', 'SpeedST', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'Position', 'SpeedI1_delta', 'SpeedST_delta', 'LapInStint', 'SpeedI1_roll3']
X_train: (25756, 14), y_train: (25756, 1)
X_test:  (13327, 14),  y_test:  (13327, 1)
Pit laps in train: 12878
Pit laps in test:  357


In [136]:
# Checking if any of the features being used are leaking
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Test each feature individually
for i, name in enumerate(feature_cols):
    rf = RandomForestClassifier(n_estimators=10, random_state=42)
    rf.fit(X_train[:, i].reshape(-1,1), y_train.ravel())
    acc = accuracy_score(y_test.ravel(), rf.predict(X_test[:, i].reshape(-1,1)))
    print(f"{name}: {acc:.4f}")

# If the output of one feature is giving ~99%, then it is leaking to the model what laps are pit laps

Driver: 0.9732
LapNumber: 0.8769
SpeedI1: 0.9692
SpeedI2: 0.9523
SpeedST: 0.9710
Compound: 0.9732
TyreLife: 0.9556
FreshTyre: 0.7706
Team: 0.9732
Position: 0.9732
SpeedI1_delta: 0.9718
SpeedST_delta: 0.9716
LapInStint: 0.9554
SpeedI1_roll3: 0.9626
